#  RAG 체인 구성 - Naïve RAG 구현

### **학습 목표:**
1. Retriever의 개념과 역할을 이해한다
2. 벡터 저장소에서 다양한 검색 방법(Top-K, Threshold, MMR)을 활용할 수 있다
3. LangChain을 사용하여 RAG 파이프라인을 구성할 수 있다
4. Gradio를 활용한 스트리밍 RAG 챗봇을 구현할 수 있다

### **실습 자료**: 
- data/transformer.pdf

---

# 환경 설정 및 준비

- 필수 라이브러리: langchain-chroma, langchain-community, faiss-cpu, langchain-openai, gradio
- 환경변수: OPENAI_API_KEY 설정 필요


`(1) Env 환경변수`

In [1]:
import os
import warnings

# Tokenizers 병렬 처리 경고 억제
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 경고 억제 (선택사항)
warnings.filterwarnings('ignore', category=UserWarning)

from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [2]:
import os
from glob import glob

`(3) 문서 로드`

In [3]:
from langchain_community.document_loaders import PyPDFLoader

# PDF 로더 초기화
pdf_loader = PyPDFLoader('./data/transformer.pdf')

# 동기 로딩
pdf_docs = pdf_loader.load()
print(f'PDF 문서 개수: {len(pdf_docs)}')

C:\Users\JSPark\AppData\Local\Temp\ipykernel_41324\3020463323.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
e:\sw\dev\ai\modu_llm7\faq_bot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PDF 문서 개수: 15


`(4) 텍스트 분할`

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

# Hugging Face의 임베딩 모델 생성
embeddings_huggingface = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

# 토크나이저 직접 접근
tokenizer = embeddings_huggingface._client.tokenizer

# 토크나이저를 사용한 예시
text = "테스트 텍스트입니다."
tokens = tokenizer(text)
print(tokens)

# 토크나이저 설정 확인
print(tokenizer.model_max_length)  # 최대 토큰 길이
print(tokenizer.vocab_size)        # 어휘 크기

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 20524.29it/s]


{'input_ids': [0, 153924, 239355, 5826, 5, 2], 'attention_mask': [1, 1, 1, 1, 1, 1]}
8192
250002


In [5]:
# 토큰 수를 계산하는 함수
def count_tokens(text):
    return len(tokenizer(text)['input_ids'])

# 토큰 수 계산
text = "테스트 텍스트입니다."
print(count_tokens(text))

6


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할기 생성
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,                      
    chunk_overlap=100,           
    length_function=count_tokens,         # 토큰 수를 기준으로 분할
    separators=["\n\n", "\n",],   # 구분자 - 재귀적으로 순차적으로 적용 
)

# 텍스트 분할
chunks = text_splitter.split_documents(pdf_docs)
print(f"생성된 텍스트 청크 수: {len(chunks)}")
print(f"각 청크의 길이: {list(len(chunk.page_content) for chunk in chunks)}")
print(f"각 청크의 토큰 수: {list(count_tokens(chunk.page_content) for chunk in chunks)}")

생성된 텍스트 청크 수: 38
각 청크의 길이: [1378, 1796, 1831, 1857, 1292, 1609, 503, 1555, 1278, 1365, 1608, 833, 1416, 1679, 999, 1764, 1604, 539, 1219, 1645, 926, 1213, 1688, 716, 1409, 1626, 624, 1411, 1438, 914, 1496, 1340, 847, 812, 470, 438, 470, 441]
각 청크의 토큰 수: [336, 415, 405, 419, 327, 424, 127, 389, 294, 382, 412, 205, 419, 417, 226, 419, 395, 149, 390, 400, 221, 356, 411, 181, 394, 405, 188, 424, 400, 278, 423, 413, 252, 178, 128, 115, 128, 111]


In [7]:
# 청크의 텍스트 확인
print(chunks[2].page_content)

1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks
in particular, have been firmly established as state of the art approaches in sequence modeling and
transduction problems such as language modeling and machine translation [ 35, 2, 5]. Numerous
efforts have since continued to push the boundaries of recurrent language models and encoder-decoder
architectures [38, 24, 15].
Recurrent models typically factor computation along the symbol positions of the input and output
sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden
states ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently
sequential nature precludes parallelization within training examples, which becomes critical at longer
sequence lengths, as memory constraints limit batching across examples. Recent work has achieved
significant improvements in computational efficiency through facto

# 벡터 저장소 기반 RAG 검색기 (Retriever)



`(1) 벡터 저장소 초기화`
- chroma 사용
- cosine distance 기준으로 인덱싱 

In [8]:
from langchain_chroma import Chroma

# Chroma 벡터 저장소 생성하기
chroma_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_huggingface,    # huggingface 임베딩 사용
    collection_name="db_transformer_cosine",    # 컬렉션 이름
    persist_directory="./chroma_db",
    collection_metadata = {'hnsw:space': 'cosine'}, # l2, ip, cosine 중에서 선택 
)

# 현재 저장된 컬렉션 데이터 확인
chroma_db.get()

{'ids': ['647f111f-6b38-4987-a363-6ba344104c31',
  '3f04ab93-f461-4010-8a1c-4b85a2b3aa0e',
  '342d99fd-0ea8-4d16-ab3c-dce39210037d',
  'ad851d65-13ed-4c97-b921-eb3ccd7be199',
  '9340d429-6d9c-4adc-8873-2e0a658d8fbb',
  '17022637-9fbf-4335-89e1-6e8d415d6ae1',
  '589e7e84-0e45-4d93-838b-ba34386af1cb',
  '97b86529-932f-42f6-a0e8-079e8626f33b',
  'ca2b41ee-ce5c-4dcd-8029-120ff58bb72f',
  '0c675460-7086-41a0-83cc-9b1d58795c6b',
  'f6c9b70e-21fd-40b9-a802-5bcd8a742a27',
  '1fec248f-2412-4ccc-bf6b-2c5d66247baf',
  '5de84653-2751-4ff9-a5f8-e271f5c08561',
  '752ad1e2-7cbf-40e2-89e0-e9add657a111',
  '78c62fae-f91c-43e2-94ad-95bf4c3f9197',
  '936cd124-2334-401f-ab24-5229ff74e785',
  'aeff5553-8936-4f70-b6d4-a2397892829c',
  'f3f541c3-67ec-4365-b538-289cd08747c2',
  '660adcc7-746e-4c80-9504-58f4529671e1',
  '68f9578a-7e54-4d8f-a57c-243f265b4cd1',
  'f1104041-609f-4bb3-b14f-aa13ec900300',
  '0d81b04a-2993-4202-a826-5621ddc2e641',
  'e83aafb1-a621-4026-af61-d1c6ff829194',
  '88fdb532-7318-4d8c-b90f-

In [9]:
chroma_db._collection.count()

38

`(2) Top K`

In [10]:
chroma_k_retriever = chroma_db.as_retriever(
    search_kwargs={"k": 2},
)

query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
retrieved_docs = chroma_k_retriever.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"-{i}-\n{doc.page_content[:100]}...{doc.page_content[-100:]} [출처: {doc.metadata['source']}]")
    print("-" * 100)

쿼리: 대표적인 시퀀스 모델은 어떤 것들이 있나요?
검색 결과:
-1-
1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural...he Transformer allows for significantly more parallelization and can reach a new state of the art in [출처: ./data/transformer.pdf]
----------------------------------------------------------------------------------------------------
-2-
In contrast to RNN sequence-to-sequence models [37], the Transformer outperforms the Berkeley-
Parse... 2016.
[2] Dzmitry Bahdanau, Kyunghyun Cho, and Yoshua Bengio. Neural machine translation by jointly [출처: ./data/transformer.pdf]
----------------------------------------------------------------------------------------------------


`(3) 임계값 지정`
- Similarity score threshold (기준 스코어 이상인 문서를 대상으로 추출)

In [13]:
from langchain_community.utils.math import cosine_similarity

chroma_threshold_retriever = chroma_db.as_retriever(
    search_type='similarity_score_threshold',       # cosine 유사도
    search_kwargs={'score_threshold': 0.5, 'k':5},  # 0.5 이상인 문서를 추출
)

query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
retrieved_docs = chroma_threshold_retriever.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
for i, doc in enumerate(retrieved_docs, 1):
    score = cosine_similarity(
        [embeddings_huggingface.embed_query(query)], 
        [embeddings_huggingface.embed_query(doc.page_content)]
        )[0][0]
    print(f"-{i}-\n{doc.page_content[:100]}...{doc.page_content[-100:]} [유사도: {score}]")
    print("-" * 100)

쿼리: 대표적인 시퀀스 모델은 어떤 것들이 있나요?
검색 결과:
-1-
1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural...he Transformer allows for significantly more parallelization and can reach a new state of the art in [유사도: 0.5069073129456894]
----------------------------------------------------------------------------------------------------
-2-
In contrast to RNN sequence-to-sequence models [37], the Transformer outperforms the Berkeley-
Parse... 2016.
[2] Dzmitry Bahdanau, Kyunghyun Cho, and Yoshua Bengio. Neural machine translation by jointly [유사도: 0.5020666736725685]
----------------------------------------------------------------------------------------------------


`(4) MMR(Maximal Marginal Relevance) 검색`

In [14]:
# MMR - 다양성 고려
chroma_mmr = chroma_db.as_retriever(
    search_type='mmr',
    search_kwargs={
        'k': 3,                 # 최종적으로 반환할 문서의 수
        'fetch_k': 8,           # 유사도 기준으로 먼저 가져올 후보 문서 수 (fetch_k >= k 권장)
        'lambda_mult': 0.5,     # 유사도와 다양성의 균형 (0=최대 다양성, 1=최대 유사도, 기본값=0.5)
        # lambda_mult가 낮을수록 서로 다른 내용의 문서를, 높을수록 쿼리와 유사한 문서를 우선 선택
        },
)


query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
retrieved_docs = chroma_mmr.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
for i, doc in enumerate(retrieved_docs, 1):
    score = cosine_similarity(
        [embeddings_huggingface.embed_query(query)], 
        [embeddings_huggingface.embed_query(doc.page_content)]
        )[0][0]
    print(f"-{i}-\n{doc.page_content[:100]}...{doc.page_content[-100:]} [유사도: {score}]")
    print("-" * 100)

쿼리: 대표적인 시퀀스 모델은 어떤 것들이 있나요?
검색 결과:
-1-
1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural...he Transformer allows for significantly more parallelization and can reach a new state of the art in [유사도: 0.5069073129456894]
----------------------------------------------------------------------------------------------------
-2-
Table 1: Maximum path lengths, per-layer complexity and minimum number of sequential operations
for ...ng
corresponds to a sinusoid. The wavelengths form a geometric progression from 2π to 10000 · 2π. We [유사도: 0.47923338150208217]
----------------------------------------------------------------------------------------------------
-3-
from our models and present and discuss examples in the appendix. Not only do individual attention
h..., according to the formula:
lrate = d−0.5
model · min(step_num−0.5, step_num · warmup_steps−1.5) (3) [유사도: 0.4709169449523962]
-----------------------------------------------------------

`(5) metadata 필터링 검색`

In [15]:
# 메타데이터 확인
chunks[0].metadata

{'producer': 'pdfTeX-1.40.25',
 'creator': 'LaTeX with hyperref',
 'creationdate': '2024-04-10T21:11:43+00:00',
 'author': '',
 'keywords': '',
 'moddate': '2024-04-10T21:11:43+00:00',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
 'subject': '',
 'title': '',
 'trapped': '/False',
 'source': './data/transformer.pdf',
 'total_pages': 15,
 'page': 0,
 'page_label': '1'}

In [16]:
# 문서 객체의 metadata를 이용한 필터링
chrom_metadata = chroma_db.as_retriever(
    search_kwargs={
        'filter': {'source': './data/transformer.pdf'},
        'k': 5, 
        }
)

query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
retrieved_docs = chrom_metadata.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"-{i}-\n{doc.page_content} [출처: {doc.metadata['source']}]")
    print("-" * 100)

쿼리: 대표적인 시퀀스 모델은 어떤 것들이 있나요?
검색 결과:
-1-
1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks
in particular, have been firmly established as state of the art approaches in sequence modeling and
transduction problems such as language modeling and machine translation [ 35, 2, 5]. Numerous
efforts have since continued to push the boundaries of recurrent language models and encoder-decoder
architectures [38, 24, 15].
Recurrent models typically factor computation along the symbol positions of the input and output
sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden
states ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently
sequential nature precludes parallelization within training examples, which becomes critical at longer
sequence lengths, as memory constraints limit batching across examples. Recent work has achieved
significant improvements i

`(6) page_content 본문 필터링 검색`

In [17]:
# page_content를 이용한 필터링
chroma_content = chroma_db.as_retriever(
    search_kwargs={
        'k': 2,
        'where_document': {'$contains': 'recurrent'},
        }
)

query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
retrieved_docs = chroma_content.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"-{i}-\n{doc.page_content} [출처: {doc.metadata['source']}]")
    print("-" * 100)

쿼리: 대표적인 시퀀스 모델은 어떤 것들이 있나요?
검색 결과:
-1-
1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks
in particular, have been firmly established as state of the art approaches in sequence modeling and
transduction problems such as language modeling and machine translation [ 35, 2, 5]. Numerous
efforts have since continued to push the boundaries of recurrent language models and encoder-decoder
architectures [38, 24, 15].
Recurrent models typically factor computation along the symbol positions of the input and output
sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden
states ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently
sequential nature precludes parallelization within training examples, which becomes critical at longer
sequence lengths, as memory constraints limit batching across examples. Recent work has achieved
significant improvements i

# [실습 프로젝트] Naive RAG 구현 

- 각 단계별 지시사항에 따라 코드를 완성하세요. 
- 제시된 지시사항과 LangChain 문서를 참조하여 시스템을 구성합니다. 

In [3]:
# # 3단계: 문서 관리

# # 새로운 문서 추가
# new_doc = Document(
#     page_content="쿠버네티스 클러스터 운영 가이드",
#     metadata={"type": "tutorial", "author": "김미영"}
# )
# new_id = str(uuid.uuid4())
# practice_db.add_documents(documents=[new_doc], ids=[new_id])
# print(f"새 문서 추가 완료: {new_id}")

# # 특정 문서 삭제 (첫 번째 문서 삭제)
# delete_id = doc_ids[0]
# practice_db.delete(ids=[delete_id])
# print(f"문서 삭제 완료: {delete_id}")

### 문서 설정

'./data/investment/6_3_유안타증권_20260612_market_554186000.pdf'
'./data/investment/냉정하게 볼 필요가 있는 금리 흐름_650022.pdf'

In [32]:
from langchain_community.document_loaders import PyPDFLoader

# PDF 로더 초기화
#pdf_loader = PyPDFLoader('./data/da/DA_모델링.pdf')
pdf_loader = PyPDFLoader('./data/6_3_유안타증권_20260612_market_554186000.pdf')

# 동기 로딩
pdf_docs = pdf_loader.load()
print(f'PDF 문서 개수: {len(pdf_docs)}')

PDF 문서 개수: 19


In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 재귀적 텍스트 분할기 초기화
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,             # 청크 크기  
    chunk_overlap=200,           # 청크 중 중복되는 부분 크기
    length_function=len,         # 글자 수를 기준으로 분할
    separators=["\n\n",  "\n", " ", ""],  # 구분자 - 재귀적으로 순차적으로 적용 
)

chunks = text_splitter.split_documents(pdf_docs)

In [34]:
print(f"생성된 텍스트 청크 수: {len(chunks)}")
print(f"각 청크의 길이: {list(len(chunk.page_content) for chunk in chunks)}")
print()

print("--- 첫 번째 청크 샘플 ---")
print(chunks[0].page_content)
print(chunks[0].metadata)

생성된 텍스트 청크 수: 21
각 청크의 길이: [87, 954, 416, 239, 884, 935, 776, 706, 645, 835, 637, 537, 727, 981, 496, 479, 735, 521, 578, 499, 2]

--- 첫 번째 청크 샘플 ---
12 June, 2026 Yuanta Research
[유안타증권 주식시황 Weekly]
KOSPI가 무릎을 꿇었던 건 추진력을 얻기 위함이다 (6월 3주)
{'producer': 'Microsoft® PowerPoint® LTSC', 'creator': 'Microsoft® PowerPoint® LTSC', 'creationdate': '2026-06-12T00:54:15+09:00', 'moddate': '2026-06-12T00:54:15+09:00', 'source': './data/6_3_유안타증권_20260612_market_554186000.pdf', 'total_pages': 19, 'page': 0, 'page_label': '1'}


`(1) 벡터 저장소 설정`
- HuggingFace에서 지원하는 BAAI/bge-m3 임베딩 모델을 사용하여 문서를 벡터화
- FAISS DB를 벡터 스토어로 사용 (IndexFlatL2 사용: 유클리드 거리)

In [35]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings  

# Hugging Face의 임베딩 모델 생성
# 힌트: HuggingFaceEmbeddings(model_name="BAAI/bge-m3") 사용
embeddings_model = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

# 임베딩 차원 확인
embedding = embeddings_model.embed_query("test")
print(f"임베딩 차원: {len(embedding)}")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 26992.77it/s]


임베딩 차원: 1024


In [36]:
# Ollama 임베딩 모델을 사용한 FAISS 벡터 저장소 생성
import faiss 
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

# FAISS 인덱스 초기화 (유클리드 거리 사용)
dim = 1024  # 임베딩 차원
faiss_index = faiss.IndexFlatL2(dim)  

# FAISS 벡터 저장소 생성
faiss_db = FAISS(
    embedding_function=embeddings_model,
    index=faiss_index,           # 벡터 검색을 위한 데이터 구조를 정의
    docstore=InMemoryDocstore(), # 문서 저장소 객체를 지정 - 문서의 원본 내용과 메타데이터를 보관
    index_to_docstore_id={},     # 인덱스와 문서 간의 연결을 관리 (매핑 딕셔너리)
)

# 저장된 문서의 갯수 확인
print(faiss_db.index.ntotal)

0


In [44]:
print(faiss_db.index.ntotal)

21


In [37]:
import uuid

# 문서 id 생성 (청크 개수 만큼)
doc_ids = [str(uuid.uuid4()) for _ in range(len(chunks))]

# 문서를 벡터 저장소에 저장
# 힌트: faiss_db.add_documents(chunks, ids=doc_ids) 사용
added_doc_ids = faiss_db.add_documents(chunks, ids=doc_ids)

# 벡터 저장소에 저장된 문서를 확인
print(f"{len(added_doc_ids)}개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.")
print(added_doc_ids)

21개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.
['a037bc71-1024-4696-a69b-b47ec98aebb3', '9c9a2102-c422-4c61-af04-875ec29391fa', '62aecaf4-53b7-440a-86a4-8df853922828', '2c975e70-6dc4-4f09-9dc8-1b8ada3f0d10', '7c8885c8-9416-4b69-8265-2cde75a02ae2', 'daec08e4-9146-434f-a523-2b9e0e5b5c2a', 'bc214359-4454-4ef2-9234-a56720337311', 'b43180b1-f60e-434a-9b9f-0b6b2d34e363', '9e0baa9c-30ac-4ddd-9401-cf7cb1c4c015', '0d377397-d85a-4ff4-8e3f-814662b299bc', 'c03dd7b5-c7d7-42ea-9489-0900cd1d8416', 'a798c85e-8a4b-46a4-adae-d40c57faab03', '0f91baea-0ec0-4491-8364-82ad59b56c0c', 'a820472b-16a0-4f78-b52f-99d1ad58cc93', 'e8fe18b8-a07e-429c-84d8-5daae661ae14', '765b02d0-1ad7-4091-9a8e-8adbc1020f93', '7b5946ec-e433-434d-9d13-6eae222c626c', '721f4416-fa7c-4f7e-9e1e-30cfd3199b8f', '11b5f8f2-3dd3-4fc9-b412-6d6ca6f318b3', '8fea31d5-d69b-4dfb-8898-de76990cc987', 'e1d7c820-9bb8-42e7-abb2-896cd5a03cac']


`(2) 검색기 정의`
- mmr 검색으로 상위 3개 문서 검색하는 Retriever 사용
- 다양성을 높이는 설정을 사용 

In [38]:
# mmr 검색기 생성
# 힌트: faiss_db.as_retriever(search_type='mmr', search_kwargs={'k': 3, 'fetch_k': 10, 'lambda_mult': 0.3})
# lambda_mult를 낮게 설정하여 다양성을 높임
faiss_mmr_retriever = faiss_db.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 3, 'fetch_k': 10, 'lambda_mult': 0.3}
    )

In [39]:
# 검색 테스트 
query = "정규화는 무엇인가요?"
# 힌트: faiss_mmr_retriever.invoke(query) 사용
retrieved_docs = faiss_mmr_retriever.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"-{i}-\n{doc.page_content[:100]}...{doc.page_content[-100:]}")
    print("-" * 100)

쿼리: 정규화는 무엇인가요?
검색 결과:
-1-
6
외국인 기관 개인
순매수
상위
화학 상사,자본재 반도체
건강관리 소프트웨어 자동차
건설,건축관련 기계 기계
철강 화장품,의류,완구 IT가전
유틸리티 IT하드웨어 조선
순매도
상...2.8 
7.2 6.6 
8.6 
-3.5 
-8.7 
-19.0 
-25
-20
-15
-10
-5
0
5
10
15
일간 주간 월간
(천억원)
외국인
기관
개인
[KOSDAQ]
----------------------------------------------------------------------------------------------------
-2-
• 이 자료에 게재된 내용들은 본인의 의견을 정확하게 반영하고 있으며 타인의 부당한 압력이나 간섭 없이 작성되었음을 확인함. (작성자: 이재원)
• 당사는 동 자료를 전문투자자 및... 어떠한 책임도 지지 않습니다. 또한, 본 자료는 당사 투자자에게만 제공되는 자료로 당사의 동의 없이 본 자료를 무단으로 복제 전송 인용
배포하는 행위는 법으로 금지되어 있습니다.
----------------------------------------------------------------------------------------------------
-3-
13
반도체노이즈지속되며차익실현매물출회, 너무많이올랐기에많아진노이즈. 과거에도반복됐으나돌아보면매수기회였다
자료: 에프앤가이드Quantiwise, 유안타증권리서치센터
딥시크, 터보퀀...
마진·램프업 속도에 대한 실망. “AI 매출 성장 둔화”가 아니라 “기대치가 너무 앞선 상태에서 나온 차익실현”
오라클 Capex·자금조달 부담 AI 데이터센터 투자가 과도하고,
----------------------------------------------------------------------------------------------------


`(3) RAG 프롬프트 구성`

- 작성 기준: 
    - LangChain의 ChatPromptTemplate 클래스 사용
    - 변수 처리는 {context}, {question} 형식 사용
    - 답변은 한글로 출력되도록 프롬프트 작성
    
- 아래 템플릿 코드를 기반으로 다음 내용을 참고하여 작성합니다. 

    1. 프롬프트 구성요소:
        - 작업 지침
        - 컨텍스트 영역
        - 질문 영역
        - 답변 형식 가이드

    2. 작업 지침:
        - 컨텍스트 기반 답변 원칙
        - 외부 지식 사용 제한
        - 불확실성 처리 방법
        - 답변 불가능한 경우의 처리 방법

    3. 답변 형식:
        - 핵심 답변 섹션
        - 근거 제시 섹션
        - 추가 설명 섹션 (필요시)

    4. 제약사항 반영:
        - 답변은 사실에 기반해야 함
        - 추측이나 가정을 최소화해야 함
        - 명확한 근거 제시가 필요함
        - 구조화된 형태로 작성되어야 함

In [40]:
# Prompt 템플릿 (여기에 작성하세요)
from langchain_core.prompts import ChatPromptTemplate

# 시스템 지침
system_message  = """당신은 주어진 컨텍스트(Context)만을 기반으로 정직하고 정확하게 답변하는 비서입니다.
[작업 지침]
1. 반드시 아래 제공된 컨텍스트(Context) 정보만을 바탕으로 질문에 답변하십시오.
2. 외부 지식이나 학습된 임의의 정보를 동원하여 사실을 왜곡하거나 임의로 유추하지 마십시오.
3. 답변이 불확실하거나 질문에 답하기 위한 정보가 부족한 경우, 가정을 배제하고 솔직하게 모른다고 답하십시오.
4. 컨텍스트에서 질문에 대한 답을 절대 찾을 수 없는 경우, 반드시 다음과 같이 답변하십시오: "제공된 컨텍스트에서 관련 정보를 찾을 수 없습니다."

[답변 형식 가이드]
반드시 다음 구조화된 형식을 엄격히 지켜 마크다운(Markdown) 형태로 답변을 작성해 주세요.

### 1. 핵심 답변
- 질문에 대한 핵심 결론과 직접적인 답변을 명확하고 간결하게 요약하여 작성합니다.
### 2. 근거 제시
- 답변의 근거가 된 컨텍스트 내의 구체적인 문장이나 핵심 단락을 그대로 인용하거나 명확하게 밝힙니다.
### 3. 추가 설명 (필요시)
- 핵심 답변을 보완하기 위해 컨텍스트에 포함되어 있는 유용한 추가 맥락이 있을 경우에만 작성합니다. (불필요한 경우 이 섹션은 생략 가능합니다.)

[제약사항]
- 답변은 100% 사실(Fact)에만 기반해야 합니다.
- 추측이나 가정을 절대 최소화하고 배제하십시오.
- 모든 답변에는 명확한 근거(근거 제시 섹션)가 포함되어야 합니다.
- 반드시 한국어로 자연스럽게 출력되도록 하십시오."""


# 사용자가 입력할 템플릿 ({context} 및 {question} 매핑)
user_message = """
[컨텍스트]
{context}

[질문]
{question}
"""

# ChatPromptTemplate 인스턴스 생성
prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_message),
    ("human", user_message)
])


# 템플릿 출력
prompt_template.pretty_print()

================================ System Message ================================

당신은 주어진 컨텍스트(Context)만을 기반으로 정직하고 정확하게 답변하는 비서입니다.
[작업 지침]
1. 반드시 아래 제공된 컨텍스트(Context) 정보만을 바탕으로 질문에 답변하십시오.
2. 외부 지식이나 학습된 임의의 정보를 동원하여 사실을 왜곡하거나 임의로 유추하지 마십시오.
3. 답변이 불확실하거나 질문에 답하기 위한 정보가 부족한 경우, 가정을 배제하고 솔직하게 모른다고 답하십시오.
4. 컨텍스트에서 질문에 대한 답을 절대 찾을 수 없는 경우, 반드시 다음과 같이 답변하십시오: "제공된 컨텍스트에서 관련 정보를 찾을 수 없습니다."

[답변 형식 가이드]
반드시 다음 구조화된 형식을 엄격히 지켜 마크다운(Markdown) 형태로 답변을 작성해 주세요.

### 1. 핵심 답변
- 질문에 대한 핵심 결론과 직접적인 답변을 명확하고 간결하게 요약하여 작성합니다.
### 2. 근거 제시
- 답변의 근거가 된 컨텍스트 내의 구체적인 문장이나 핵심 단락을 그대로 인용하거나 명확하게 밝힙니다.
### 3. 추가 설명 (필요시)
- 핵심 답변을 보완하기 위해 컨텍스트에 포함되어 있는 유용한 추가 맥락이 있을 경우에만 작성합니다. (불필요한 경우 이 섹션은 생략 가능합니다.)

[제약사항]
- 답변은 100% 사실(Fact)에만 기반해야 합니다.
- 추측이나 가정을 절대 최소화하고 배제하십시오.
- 모든 답변에는 명확한 근거(근거 제시 섹션)가 포함되어야 합니다.
- 반드시 한국어로 자연스럽게 출력되도록 하십시오.

================================ Human Message =================================


[컨텍스트]
{context}

[질문]
{question}



`(4) RAG 체인 구성`
- LangChain의 LCEL 문법을 사용
- 검색 결과를 프롬프트의 'context'로 전달하고,
- 사용자가 입력한 질문을 그래도 프롬프트의 'question'에 전달
- LLM 설정:
    - ChatOpenAI 사용 ('gpt-4o-mini' 모델)
    - temperature: 답변의 일관성을 가져가는 설정값을 사용 
    - 기타 필요한 설정 
- 출력 파서: 문자열 부분만 출력되도록 구성

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# LLM 설정
# 힌트: ChatOpenAI(model='gpt-4o-mini', temperature=0) 사용
llm = ChatOpenAI(
    model="gpt-4.1-mini", 
    temperature=0
)

# 문서 포맷팅
def format_docs(docs):
    return "\n\n".join([f"{doc.page_content}" for doc in docs])

# RAG 체인 생성
# 힌트: {'context': faiss_mmr_retriever | format_docs, 'question': RunnablePassthrough()} | prompt | llm | StrOutputParser()
rag_chain = {'context': faiss_mmr_retriever | format_docs, 'question': RunnablePassthrough()} | prompt_template | llm | StrOutputParser()

# 체인 실행
#query = "정규화는 무엇인가요?"
query = "코스피 현황은 어떻게 되나요?"
output = rag_chain.invoke(query)

print(f"쿼리: {query}")
print("답변:")
print(output)

쿼리: 코스피의 현황은 어떻게 되나요?
답변:
### 1. 핵심 답변
- 최근 KOSPI는 6월 1주차에 약 10.1% 하락하는 큰 조정을 겪었으나, 이는 과도한 쏠림 현상의 해소와 차익실현에 따른 일시적 조정으로 해석됩니다. 반도체 수출 호조와 실적 개선 기대가 지수 하단을 지지하고 있으며, 과거 유사 급락 사례와 비교할 때 단기 내 반등 가능성이 높습니다.

### 2. 근거 제시
- "주간(6/5~6/11) KOSPI, KOSDAQ은 각각 -10.1%, -5.0% 하락했습니다. 전주 신고가 경신 이후 금리 변동성, 지정학적 갈등, AI 투자 우려가 겹치며 국내 증시도 대형주 중심 차익실현이 나타났습니다."
- "이번 조정의 명분은 전쟁·유가·금리였지만, 본질은 5월 이후 과도했던 쏠림 해소였습니다."
- "6월 1~10일 총수출은 286억달러, 반도체 수출은 111억달러로 동기간 기준 역대 최대 수준을 기록했습니다... 2분기 실적 기대는 유효합니다."
- "KOSPI 12개월 선행 EPS도 반도체 수출과 높은 동행성을 보이고 있어, 삼성전자 잠정실적을 기점으로 재상향 가능성이 큽니다."
- "2000년 이후 일일 KOSPI 지수 8% 이상 하락은 총 7차례. 지수는 2008년 금융위기 당시 시스템 리스크가 실물 경기 침체로 확산된 경우를 제외하고 모두 크게 반등했던 경험."
- "급락일 이후 10일, 30일, 90일 평균 수익률은 각각 5.5%, 6.5%, 15.3%. 현재 상황을 스태그플레이션이나 경기 침체로 보긴 어려움."

### 3. 추가 설명
- AI 투자 우려와 지정학적 리스크 등 단기적 노이즈가 있으나, 이는 수요 둔화보다는 공급 제약과 기대치 조정에 가깝다는 분석이 있습니다.
- 과거 급락 사례와 비교할 때, 이번 급락은 시스템 리스크가 실물 경기 침체로 확산된 경우가 아니므로 단기 내 반등 가능성이 높다는 점이 긍정적입니다.


`(5) Gradio 스트리밍 구현`
- ChatInterface 사용
- `chain.stream()`으로 응답을 청크 단위로 스트리밍

In [48]:
import gradio as gr
from typing import Iterator

# 스트리밍 응답 생성 함수
def get_streaming_response(message: str, history) -> Iterator[str]:
    
    # RAG Chain 실행 및 스트리밍 응답 생성
    response = ""
    for chunk in rag_chain.stream(message):
        if isinstance(chunk, str):
            response += chunk
            yield response

# Gradio 인터페이스 설정
# 힌트: gr.ChatInterface(fn=get_streaming_response, title="RAG 기반 질의응답 시스템", description="...", examples=[...])
demo = gr.ChatInterface(
    fn=get_streaming_response, 
    title="RAG 기반 질의응답 시스템", 
    description="사용자 질의에 대한 최신 정보를 답변합니다.", 
    examples=["최신 상승 주식은 무엇인가요?",
    "미국 이란 갈등은 어떻게 되고있나요?", 
    "마지막 KOSPI 지수는 어떻게 되었나요?",
    ])

# 실행
demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [45]:
# demo 실행 종료
demo.close()

Closing server running on port: 7862


In [ ]:
# RAG 문서 추가 
import uuid
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. 새 PDF 문서 로드
new_pdf_path = "./data/investment/냉정하게 볼 필요가 있는 금리 흐름_650022.pdf"
loader = PyPDFLoader(new_pdf_path)

new_pdf_docs = loader.load()

# 2. 텍스트 분할 
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

new_chunks = text_splitter.split_documents(new_pdf_docs)

# 3. 고유한 문서 ID 목록 생성
new_doc_ids = [str(uuid.uuid4()) for _ in range(len(new_chunks))]
print(f"새로 생성된 청크 개수: {len(new_chunks)}")


새로 생성된 청크 개수: 10


In [47]:
# faiss_db에 새 문서 추가
added_ids = faiss_db.add_documents(
    documents=new_chunks,
    ids=new_doc_ids
)
print(f"FAISS DB에 문서 {len(added_ids)}개 추가 완료.")
print(f"현재 총 FAISS 문서 개수: {faiss_db.index.ntotal}")

FAISS DB에 문서 10개 추가 완료.
현재 총 FAISS 문서 개수: 31


In [51]:
new_pdf_docs[0].metadata

{'producer': 'Microsoft® Word 2021',
 'creator': 'Microsoft® Word 2021',
 'creationdate': '2026-06-12T06:53:10+09:00',
 'moddate': '2026-06-12T06:53:10+09:00',
 'source': './data/냉정하게 볼 필요가 있는 금리 흐름_650022.pdf',
 'total_pages': 7,
 'page': 0,
 'page_label': '1'}

In [52]:
# RAG 문서 추가를 위한 함수 설정

import os
import uuid
from typing import List, Optional
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def add_pdf_to_vectorstore(
    file_path: str,
    vector_store,
    chunk_size: int = 1000,
    chunk_overlap: int = 200
) -> List[str]:
    """
    특정 PDF 파일을 로드, 청킹하여 벡터 저장소에 동적으로 추가합니다.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {file_path}")
        
    print(f"[{os.path.basename(file_path)}] 문서 로드 중...")
    
    # 1. 문서 로드
    loader = PyPDFLoader(file_path)
    docs = loader.load()
    
    # 2. 메타데이터 보강 (나중에 파일별 삭제나 추적이 가능하도록 파일 소스 기록)
    for doc in docs:
        doc.metadata["source_file"] = os.path.abspath(file_path)
        
    # 3. 텍스트 분할
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    chunks = text_splitter.split_documents(docs)
    
    # 4. 고유 ID 생성 및 문서 저장소 추가
    doc_ids = [str(uuid.uuid4()) for _ in range(len(chunks))]
    
    # 벡터 저장소에 추가
    added_ids = vector_store.add_documents(documents=chunks, ids=doc_ids)
    
    print(f"성공적으로 추가 완료! (생성된 청크 수: {len(added_ids)}개)")
    return added_ids

In [ ]:
# 추가 PDF 문서 로드
new_file = "./data/investment/보고서_20260611.pdf"

add_pdf_to_vectorstore(new_file, faiss_db)

[보고서_20260611.pdf] 문서 로드 중...
성공적으로 추가 완료! (생성된 청크 수: 11개)


['7ed27b2d-e798-464c-87ad-b2279f73fda7',
 'e62ffbc9-3684-4bb9-b3e0-f568453a823a',
 '0d24293a-47bf-4000-a6b0-928996e834cd',
 '42a66d97-452f-47fb-a308-469d9d804671',
 'cf7e59ae-d191-44c9-9872-688bbc447d68',
 '5128c546-ed59-4b98-a6c8-4d8e0eaab566',
 '79408097-b0e0-42c5-9c2d-373db07eb850',
 'd3a21456-8625-4e4d-bfec-4184a1806975',
 '817f1498-24cd-4019-b874-6b27d398a7de',
 'dfce5017-2e61-491c-9be1-b18e1423fae3',
 '68f98248-b6e9-420b-a017-b67ebdd39834']

In [ ]:
print(f"현재 총 FAISS 문서 개수: {faiss_db.index.ntotal}")

# 다시 gradio에서 검색 - 동적 검색 확인 완료

현재 총 FAISS 문서 개수: 42
